# Exercises

E01: train a trigram language model, i.e. take two characters as an input to predict the 3rd one. Feel free to use either counting or a neural net. Evaluate the loss; Did it improve over a bigram model?

In [2]:
import torch
import torch.nn.functional as F

# bring in data
words = open('names.txt', 'r').read().splitlines()

In [3]:
'''
step 1: create mapping dicts
'''
chars = sorted(list(set(''.join(words))))
chars.insert(0, '.')


# set up single character index
s_to_i = {s: i for i, s in enumerate(chars)}
i_to_s = {i: s for i, s in enumerate(chars)}

In [4]:
'''
step 2: create x and y
'''
def create_trigrams(words_list:list):
    xs, ys = [], []

    for w in words_list:
        chs = ['.'] + list(w) + ['.']
        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            ix1 = s_to_i[ch1]
            ix2 = s_to_i[ch2]
            ix3 = s_to_i[ch3]
            #print(ch1, ch2, ch3)
            xs.append([ix1, ix2])
            ys.append(ix3)

    xs = torch.tensor(xs) 
    ys = torch.tensor(ys)
    
    return xs, ys

In [5]:
'''
step 3: set up neural net
'''
xnn, ynn = create_trigrams(words)
num = ynn.nelement()
g = torch.Generator().manual_seed(2147483647)
W_nn = torch.randn((27*2, 27), generator=g, requires_grad=True)


In [6]:
'''
step 5: training loop
'''
n_iter_nn, lr_nn, C_nn = 250, 50, 0.0
# gradient descent
for k in range(n_iter_nn):

    # forward pass
    xenc_nn = F.one_hot(xnn, num_classes=27).float()            # input to the network: one-hot encoding
    logits_nn = xenc_nn.view(-1, 27*2) @ W_nn                   # predict log-counts
    counts_nn = logits_nn.exp()                                 # counts, equivalent to N
    probs_nn = counts_nn / counts_nn.sum(1, keepdim=True)       # probabilities for next character
    loss_nn = -probs_nn[torch.arange(num), ynn].log().mean() + C_nn*(W_nn**2).mean() 
    #print(loss_nn.item())
    if k > (n_iter_nn - 5):
        print(loss_nn.item())

    # backward pass
    W_nn.grad = None # set to zero the gradient
    loss_nn.backward()

    # update
    W_nn.data += -lr_nn * W_nn.grad
    
print(f'\naverage negative log_likelihood = {loss_nn.item():.4f}\n')

2.245729923248291
2.245682716369629
2.245635986328125
2.245589256286621

average negative log_likelihood = 2.2456



In [7]:
# try and sample
'''
step 6 try and random sample
'''
g = torch.Generator().manual_seed(2147483647+1)
generated_names = []
for i in range(5):
    out = []
    # Initializing the starting sequence
    context = [0, 0] 
    while True:
        x_encoded = F.one_hot(torch.tensor([context]), num_classes=27).float().view(1, -1)
        logits = x_encoded @ W_nn
        counts = logits.exp()
        probas = counts / counts.sum(1, keepdims=True)
        
        ix = torch.multinomial(
            probas, num_samples=1, replacement=True, generator=g
        ).item()
        # Adding the predicted character and shifting
        context = context[1:] + [ix]
        out.append(i_to_s[ix])
        if ix == 0:
            break
            
    generated_names.append(''.join(out)[:-1])

print(generated_names)


['xon', 'austy', 'ziazedeeth', 'ubaneromidann', 'or']


E02: split up the dataset randomly into 80% train set, 10% dev set, 10% test set. Train the bigram and trigram models only on the training set. Evaluate them on dev and test splits. What can you see?

In [8]:
from sklearn.model_selection import train_test_split

train_val,test = train_test_split(words, test_size=0.1, random_state=42)

val_size_relative = 0.1 / (1.0 - 0.1)
train , val = train_test_split(train_val, test_size=val_size_relative, random_state=42)


g = torch.Generator().manual_seed(2147483647)
W_nn = torch.randn((27*2, 27), generator=g, requires_grad=True)

In [9]:

train_xs, train_ys = create_trigrams(train)
test_xs, test_ys = create_trigrams(test)

n_iter_nn, lr_nn, C_nn = 250, 50, 0.0
num = train_ys.nelement()
# gradient descent
for k in range(n_iter_nn):

    # forward pass
    xenc_nn = F.one_hot(train_xs, num_classes=27).float()            # input to the network: one-hot encoding
    logits_nn = xenc_nn.view(-1, 27*2) @ W_nn                   # predict log-counts
    counts_nn = logits_nn.exp()                                 # counts, equivalent to N
    probs_nn = counts_nn / counts_nn.sum(1, keepdim=True)       # probabilities for next character
    loss_nn = -probs_nn[torch.arange(num), train_ys].log().mean() + C_nn*(W_nn**2).mean() 
    #print(loss_nn.item())
    if k > (n_iter_nn - 5):
        print(loss_nn.item())

    # backward pass
    W_nn.grad = None # set to zero the gradient
    loss_nn.backward()

    # update
    W_nn.data += -lr_nn * W_nn.grad
    
print(f'\naverage negative log_likelihood = {loss_nn.item():.4f}\n')

2.245574474334717
2.2455272674560547
2.245480537414551
2.245434284210205

average negative log_likelihood = 2.2454



In [ ]:
# validation error
# to evaluate, we just do another forward pass
val_xs, val_ys = create_trigrams(val)

xenc_val = F.one_hot(val_xs, num_classes=27).float()
logits_nn = xenc_val.view(-1, 27*2) @ W_nn
counts_nn = logits_nn.exp()
probs_nn = counts_nn / counts_nn.sum(1, keepdim=True)
loss_nn = -probs_nn[torch.arange(val_ys.nelement()), val_ys].log().mean()
loss_nn.item()

2.244619131088257

In [ ]:
# test error
test_xs, test_ys = create_trigrams(test)

xenc_val = F.one_hot(test_xs, num_classes=27).float()
logits_nn = xenc_val.view(-1, 27*2) @ W_nn
counts_nn = logits_nn.exp()
probs_nn = counts_nn / counts_nn.sum(1, keepdim=True)
loss_nn = -probs_nn[torch.arange(test_ys.nelement()), test_ys].log().mean()
loss_nn.item()

2.2535765171051025

E04: we saw that our 1-hot vectors merely select a row of W, so producing these vectors explicitly feels wasteful. Can you delete our use of F.one_hot in favor of simply indexing into rows of W?

In [ ]:
# instead of matrix multiplying with a one hot encoding, we can just slice the weights matrix according to the binary encodings anyway

first_col = train_xs[:, 0] # this returns the first column of the xs, which represents the index of the first character in the bigram
second_col_shifted = train_xs[:, 1] + 27 # this returns the second column of xs, which is the index of the second character in the bigram - we adjust by 27 so it aligns with the 28-54 inputs in the neural net
logits_idx = W_nn[first_col] + W_nn[second_col_shifted]
counts_idx = logits_idx.exp()
probs = counts_idx / counts_idx.sum(1, keepdim=True)

loss = -probs[torch.arange(train_ys.nelement()), train_ys].log().mean()
loss # same as the matrix multiplication


tensor(2.2454, grad_fn=<NegBackward0>)